# ParkSight Quickstart

End-to-end walkthrough: **address → parking features → satellite tiles → stall counts → interactive map**.

Estimated time: ~5 minutes.

## 1. Install & import

In [1]:
# Uncomment to install (run once):
# %pip install -r ../requirements.txt

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from parksight import fetch, count, viz, imagery, utils
import matplotlib.pyplot as plt
import numpy as np

print("ParkSight loaded!")

ParkSight loaded!


## 2. Configure address & radius

In [ ]:
ADDRESS = "222 Third St, Cambridge, MA"
RADIUS = 300  # metres

## 3. Fetch parking features from OSM

In [3]:
gdf, (lat, lon) = fetch.get_parking_data(ADDRESS, dist=RADIUS)
print(f"Centre: ({lat:.5f}, {lon:.5f})")
print(f"Found {len(gdf)} parking features")
gdf.head()

(33.7760948, -84.3988077)
Centre: (33.77609, -84.39881)
Found 8 parking features


geometry  amenity  \
element  id                                                                     
relation 301779    POLYGON ((-84.3993 33.77403, -84.39932 33.7737...  parking   
         312652    POLYGON ((-84.3993 33.7759, -84.3993 33.77574,...  parking   
way      43087052  POLYGON ((-84.39692 33.77775, -84.39691 33.777...  parking   
         43143609  POLYGON ((-84.39679 33.7771, -84.39672 33.7771...  parking   
         43332751  POLYGON ((-84.40122 33.77387, -84.40117 33.774...  parking   

                  location  parking  \
element  id                           
relation 301779        NaN      NaN   
         312652        NaN  surface   
way      43087052      yes  surface   
         43143609      yes      NaN   
         43332751      NaN      yes   

                                                            source wheelchair  \
element  id                                                                     
relation 301779                                                NaN        NaN   
         312652    Georgia Tech (http://www.arch.gatech.edu/cgis/)        NaN   
way      43087052  Georgia Tech (http://www.arch.gatech.edu/cgis/)        NaN   
         43143609  Georgia Tech (http://www.arch.gatech.edu/cgis/)        yes   
         43332751  Georgia Tech (http://www.arch.gatech.edu/cgis/)        NaN   

                  building gatech:BLDG_NUM           gatech:BLDG_TYPE  \
element  id                                                             
relation 301779        NaN             NaN                        NaN   
         312652        NaN             NaN                        NaN   
way      43087052      NaN             NaN                        NaN   
         43143609      NaN             NaN                        NaN   
         43332751      yes              54  Parking / Visitor Parking   

                  gatech:PRKG_NUM  ... name_1 oneway parking:right  \
element  id                        ...                               
relation 301779               NaN  ...    NaN    NaN           NaN   
         312652               W21  ...    NaN    NaN           NaN   
way      43087052             NaN  ...    NaN    NaN           NaN   
         43143609             NaN  ...    NaN    NaN           NaN   
         43332751             W02  ...    NaN    NaN           NaN   

                  parking:right:orientation sidewalk:both surface  \
element  id                                                         
relation 301779                         NaN           NaN     NaN   
         312652                         NaN           NaN     NaN   
way      43087052                       NaN           NaN     NaN   
         43143609                       NaN           NaN     NaN   
         43332751                       NaN           NaN     NaN   

                  turn:lanes:backward layer          type  \
element  id                                                 
relation 301779                   NaN   NaN  multipolygon   
         312652                   NaN   NaN  multipolygon   
way      43087052                 NaN   NaN           NaN   
         43143609                 NaN   NaN           NaN   
         43332751                 NaN   NaN           NaN   

                                                    lat_lon  
element  id                                                  
relation 301779    (33.773878788066554, -84.39962191294096)  
         312652     (33.77711531958579, -84.39951516193152)  
way      43087052   (33.77766934724165, -84.39710509550794)  
         43143609    (33.77744607403544, -84.3965711125875)  
         43332751    (33.77415838466646, -84.4006057570323)  

[5 rows x 35 columns]

## 4. Visualise on interactive map

In [4]:
m = viz.create_parking_map(lat, lon, gdf, radius=RADIUS)
m

## 5. Fetch a satellite tile for one feature

In [ ]:
# Pick the first polygon feature
gdf_3857 = gdf.to_crs(epsg=3857)
polygons = gdf_3857[gdf_3857.geometry.geom_type == "Polygon"]

if len(polygons) > 0:
    sample_geom = polygons.iloc[0].geometry
    tile = fetch.get_satellite_tile(sample_geom)
    plt.figure(figsize=(6, 6))
    plt.imshow(tile)
    plt.title("Satellite tile")
    plt.axis("off")
    plt.show()
else:
    print("No polygon features found — try a larger radius.")

## 6. Run CV baseline counting on all features

In [ ]:
counts = []
for _, row in gdf_3857.iterrows():
    try:
        c = count.count_edges(row.geometry)
    except Exception as e:
        print(f"Skipped {row.name}: {e}")
        c = 0
    counts.append(c)

gdf["count"] = counts
total = sum(counts)
print(f"\nCV baseline total: {total} estimated stalls")
gdf[["count"]].head(10)

## 7. Show CV pipeline steps (educational)

In [ ]:
if len(polygons) > 0:
    steps = count.visualize_pipeline(polygons.iloc[0].geometry)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, (name, img) in zip(axes, steps.items()):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(img)
        ax.set_title(name)
        ax.axis("off")
    plt.suptitle("CV Pipeline Steps", fontsize=14)
    plt.tight_layout()
    plt.show()

## 8. Run ML baseline (Grounding DINO) on a sample tile

This uses a zero-shot object detection model — no training required.
The first run will download the model weights (~350 MB).

In [ ]:
from parksight.detect import ParkingDetector

detector = ParkingDetector()

if len(polygons) > 0:
    tile = fetch.get_satellite_tile(polygons.iloc[0].geometry)
    result = detector.detect(tile)
    print(f"Detected {len(result.boxes)} objects")

    annotated = detector.annotate(tile, result)
    plt.figure(figsize=(8, 8))
    plt.imshow(annotated)
    plt.title(f"Grounding DINO: {len(result.boxes)} detections")
    plt.axis("off")
    plt.show()

## 9. Compare CV vs ML baseline results

In [ ]:
if len(polygons) > 0:
    sample_geom = polygons.iloc[0].geometry
    cv_count = count.count_edges(sample_geom)
    ml_count = detector.count_spots(fetch.get_satellite_tile(sample_geom))

    print(f"CV baseline:  {cv_count} stalls")
    print(f"ML baseline:  {ml_count} detections")
    print(f"Difference:   {abs(cv_count - ml_count)}")

## 10. Baseline accuracy vs ground truth

| Location | Ground Truth | CV Baseline | ML Baseline |
|----------|-------------|-------------|-------------|
| TBD      | TBD         | TBD         | TBD         |
| TBD      | TBD         | TBD         | TBD         |
| TBD      | TBD         | TBD         | TBD         |

*Fill in once you have ground-truth stall counts.*

## 11. Final map with counts

In [ ]:
m = viz.create_parking_map(lat, lon, gdf, radius=RADIUS, total_count=total)
m

---

## Your Turn!

The baselines above are intentionally simple. Here are some ideas to beat them:

### Track 1: Better CV
- Tune `config.json` parameters (blur kernel, Canny thresholds, Hough params)
- Try adaptive thresholding instead of Canny
- Use contour-based counting instead of line counting

### Track 2: Better ML
- Fine-tune Grounding DINO on parking-specific data (see `02_improve_counting.ipynb`)
- Try YOLOv8/v9 with a parking dataset (ParkSeg12k, APKLOT)
- Use SAM (Segment Anything) to segment individual stalls
- Ensemble CV + ML predictions

### Track 3: Novel approaches
- Density estimation (count without detecting individual spots)
- Semantic segmentation → pixel ratio → count
- Multi-scale tiling for large lots
- Temporal analysis (occupied vs empty stalls)